In this files we want to analyze the results of the secda_apps_evaluation_suite
app 1: Inference Diff

In [3]:
# code description

# take input of folder name as a list
# in each folder check how many .txt files are there
# in each .txt file look for a keyword and its corresponding value
# create csv file within each folder where we will have two columns: name of txt file, keyword value
import os
import csv
import re

def extract_keyword_value(file_path, keyword):
    """Extracts value of given keyword (e.g., avg_error=91.554) from file."""
    with open(file_path, "r") as f:
        text = f.read()

        # Regex: match keyword= then capture floats including scientific notation
        pattern = rf"{keyword}\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    return None


def process_folders(folders, keyword):
    for folder in folders:
        if not os.path.exists(folder):
            print(f"❌ Folder not found: {folder}")
            continue

        txt_files = [f for f in os.listdir(folder) if f.endswith(".txt")]
        if not txt_files:
            print(f"⚠️ No .txt files in {folder}")
            continue

        output_file = os.path.join(folder, f"{keyword}_summary.csv")

        with open(output_file, "w", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["filename", keyword])  # header

            for txt in txt_files:
                txt_path = os.path.join(folder, txt)
                value = extract_keyword_value(txt_path, keyword)
                writer.writerow([txt, value if value else "N/A"])

        print(f"✅ Saved summary for {folder} → {output_file}")

# Example usage
# folders = ["results/id_test1_pynq_2025_09_02_16_56",
#            "results/id_test2_pynq_2025_09_02_17_11"]  # list of folders
# folders = ["results/id_test3_pynq_2025_09_02_20_57",
#            "results/id_test4_pynq_2025_09_02_21_02"]  # list of folders
folders = ["results/id_test1_kria_2025_09_02_21_24",
           "results/id_test2_kria_2025_09_02_21_28"]  # list of folders
# folders = ["results/id_test3_kria_2025_09_02_21_15",
#            "results/id_test4_kria_2025_09_02_21_19"]  # list of folders
keyword = "avg_error"  # the keyword you want to extract
process_folders(folders, keyword)

✅ Saved summary for results/id_test1_kria_2025_09_02_21_24 → results/id_test1_kria_2025_09_02_21_24/avg_error_summary.csv
✅ Saved summary for results/id_test2_kria_2025_09_02_21_28 → results/id_test2_kria_2025_09_02_21_28/avg_error_summary.csv


In [4]:
# in the given folder list check avg_error_summary.csv exists or not
# if does not exists printout which folders does not have it
# compare the avg_error values
import os
import pandas as pd

def check_missing_csv(folders, keyword):
    """Check if keyword_summary.csv exists in each folder."""
    missing = []
    csv_files = {}
    for folder in folders:
        csv_path = os.path.join(folder, f"{keyword}_summary.csv")
        if not os.path.exists(csv_path):
            print(f"❌ Missing: {csv_path}")
            missing.append(folder)
        else:
            csv_files[folder] = csv_path
    return missing, csv_files


def compare_runs(folders, keyword="avg_error", tolerance=0.0):
    """Compare avg_error values across multiple runs."""
    missing, csv_files = check_missing_csv(folders, keyword)

    # Sanity check: each folder must have its CSV
    if missing:
        print("\n📌 These folders are missing CSV files:")
        for f in missing:
            print("   ", f)
        return None

    dfs = []
    for folder, csv_path in csv_files.items():
        run_name = os.path.basename(folder.rstrip("/"))
        df = pd.read_csv(csv_path)
        df[keyword] = pd.to_numeric(df[keyword], errors="coerce")
        df.rename(columns={keyword: f"{keyword}_{run_name}"}, inplace=True)
        dfs.append(df)

    # Merge all dataframes on filename
    merged = dfs[0]
    for df in dfs[1:]:
        merged = pd.merge(merged, df, on="filename", how="outer")

    # Compare against the first run
    ref_col = f"{keyword}_{os.path.basename(folders[0].rstrip('/'))}"
    for folder in folders[1:]:
        run_col = f"{keyword}_{os.path.basename(folder.rstrip('/'))}"
        match_col = f"matched_vs_{os.path.basename(folder.rstrip('/'))}"
        merged[match_col] = (merged[ref_col] - merged[run_col]).abs() <= tolerance

    # Save merged comparison into a master CSV
    out_file = os.path.join(os.path.dirname(list(csv_files.values())[0]), f"{keyword}_comparison.csv")
    merged.to_csv(out_file, index=False)

    print(f"\n✅ Saved merged CSV with {len(folders)} runs to {out_file}")
    print(merged.head())

    return merged


# Example usage
# folders = ["results/id_test1_pynq_2025_09_02_16_56",
#            "results/id_test2_pynq_2025_09_02_17_11"]  # list of folders
# folders = ["results/id_test3_pynq_2025_09_02_20_57",
#            "results/id_test4_pynq_2025_09_02_21_02"]  # list of folders
folders = ["results/id_test1_kria_2025_09_02_21_24",
           "results/id_test2_kria_2025_09_02_21_28"]  # list of folders
# folders = ["results/id_test3_kria_2025_09_02_21_15",
#            "results/id_test4_kria_2025_09_02_21_19"]  # list of folders
compare_runs(folders, "avg_error", tolerance=1e-6)


✅ Saved merged CSV with 2 runs to results/id_test1_kria_2025_09_02_21_24/avg_error_comparison.csv
                                            filename  \
0  inference_diff_VMRPP_KRIA_GEMM8_250M_vm_delega...   
1  inference_diff_VMRPP_KRIA_GEMM8_250M_vm_delega...   
2  inference_diff_VMRPP_KRIA_SH_APOT_OPT_GEMM8_25...   
3  inference_diff_VMRPP_KRIA_SH_APOT_OPT_GEMM8_25...   
4  inference_diff_VMRPP_KRIA_SH_MSQ_OPT_GEMM8_250...   

   avg_error_id_test1_kria_2025_09_02_21_24  \
0                                     0.000   
1                                     0.000   
2                                    91.866   
3                                    90.617   
4                                    90.178   

   avg_error_id_test2_kria_2025_09_02_21_28  \
0                                     0.000   
1                                     0.000   
2                                    91.866   
3                                    90.617   
4                                    90.178   

,filename,avg_error_id_test1_kria_2025_09_02_21_24,avg_error_id_test2_kria_2025_09_02_21_28,matched_vs_id_test2_kria_2025_09_02_21_28
0,inference_diff_VMRPP_KRIA_GEMM8_250M_vm_delega...,0.000,0.000,True
1,inference_diff_VMRPP_KRIA_GEMM8_250M_vm_delega...,0.000,0.000,True
2,inference_diff_VMRPP_KRIA_SH_APOT_OPT_GEMM8_25...,91.866,91.866,True
3,inference_diff_VMRPP_KRIA_SH_APOT_OPT_GEMM8_25...,90.617,90.617,True
4,inference_diff_VMRPP_KRIA_SH_MSQ_OPT_GEMM8_250...,90.178,90.178,True
5,inference_diff_VMRPP_KRIA_SH_MSQ_OPT_GEMM8_250...,89.644,89.644,True
6,inference_diff_VMRPP_KRIA_SH_QK_GEMM8_250M_vm_...,97.817,97.817,True
7,inference_diff_VMRPP_KRIA_SH_QK_GEMM8_250M_vm_...,92.068,92.068,True
